
**Created by:**

__Viktor Varga__

<br>
<img src="../_files/nb_im/ik_logo.png" style="display:inline-block" width='40%'>
<hr>

# **NumPy Array Programming Guide** - Chapter 3


## _Vector $\rightarrow$ scalar_ (collapse) operations on array slices


In Chapter 1 we discussed operations that compute a single value from a group of elements (_vector $\rightarrow$ scalar_ operations). Examples include sum, product, and searching for the maximum or minimum. In the previous notebook we still applied these operations to the entire array, producing a single scalar value.

In data science and machine learning it is common to apply the same operations independently to smaller slices of a larger dataset. Many NumPy functions make it possible to first slice an array automatically along one or more axes, and then **execute the function on each slice independently**. Functions that support this feature provide the `axis` parameter. With it, we can specify along which axis (or axes) we slice the array before performing the operation.

<img src='../_files/nb_im/numpy_basics-axis_reduce_ops.png' width='90%'>

<br>$\ $</br>

As the figure above shows, for _vector $\rightarrow$ scalar_ operations **elements are collapsed along the axes given in the `axis` parameter, so those axes disappear from the returned array**, hence the other name: collapse operations. If we provide all axes of an array in `axis`, the function returns a scalar.

A good example of where this is useful is computing temperature averages, maxima, minima, precipitation sums, etc. on weather time series. If the series contains hourly measurements, it can be reshaped to `(number_of_days, 24)`. Then we can compute average/min/max temperature (or precipitation sum) for each hour across all days, or compute the same statistics for each day across all hours. The only difference between these two computations is the `axis` parameter.

Below we apply summation to a 2D array of shape `(2, 3)` first along axis #0 and then along axis #1. The first operation removes axis #0 and returns an array of shape `(3,)`; the second removes axis #1 and returns an array of shape `(2,)`. If we do not provide `axis` (default `None`), summation is performed over the entire array and the result is a scalar.


In [ ]:
import numpy as np

arr2d = np.arange(6, dtype=np.int32).reshape(2,3)

print("The 2D 'arr2d' array:\n", arr2d)
print("   ... its shape is", arr2d.shape)

sum0 = np.sum(arr2d, axis=0)
print("\nnp.sum(arr2d, axis=0).shape: ", sum0.shape)
print("np.sum(arr2d, axis=0): ", sum0)

sum1 = np.sum(arr2d, axis=1)
print("\nnp.sum(arr2d, axis=1).shape: ", sum1.shape)
print("np.sum(arr2d, axis=1): ", sum1)

print("\nnp.sum(arr2d): ", np.sum(arr2d))     # scalar, default: axis = None

The 2D 'arr2d' array:
 [[0 1 2]
 [3 4 5]]
   ... its shape is (2, 3)

np.sum(arr2d, axis=0).shape:  (3,)
np.sum(arr2d, axis=0):  [3 5 7]

np.sum(arr2d, axis=1).shape:  (2,)
np.sum(arr2d, axis=1):  [ 3 12]

np.sum(arr2d):  15


If we want to apply an operation along multiple axes, one option is to call the function multiple times with different `axis` values. In that case, instead of `np.<function_name>`, it is often preferable to call the `ndarray.<function_name>` variant (for example, `arr2d.sum()` instead of `np.sum(arr2d)`), because it chains naturally and makes the code more readable.

However, for most functions that support `axis`, we can specify multiple axes at once by passing a tuple, as shown in the third case in the cell below.


In [ ]:
sum10 = np.sum(np.sum(arr2d, axis=1), axis=0)
print("np.sum(np.sum(arr2d, axis=1), axis=0): ", sum10)   # scalar

sum10nd = arr2d.sum(axis=1).sum(axis=0)
print("\narr2d.sum(axis=1).sum(axis=0): ", sum10nd)   # scalar

sum01 = np.sum(arr2d, axis=(0,1))
print("\nnp.sum(arr2d, axis=(0,1)): ", sum01)   # scalar


np.sum(np.sum(arr2d, axis=1), axis=0):  15

arr2d.sum(axis=1).sum(axis=0):  15

np.sum(arr2d, axis=(0,1)):  15


It is worth paying attention that if we use the chaining approach, we should list the axes to be reduced from the larger index to the smaller one, to avoid confusion and mistakes. After we remove axis #0 in the example below, the old axis #1 becomes axis #0 in the input of the next operation.


In [ ]:
sum10nd = arr2d.sum(axis=1).sum(axis=0)
print("arr2d.sum(axis=1).sum(axis=0): ", sum10nd)   # scalar

sum00nd = arr2d.sum(axis=0).sum(axis=0)
print("\narr2d.sum(axis=0).sum(axis=0): ", sum00nd)   # scalar, same as above, but more confusing axis parameters


arr2d.sum(axis=1).sum(axis=0):  15

arr2d.sum(axis=0).sum(axis=0):  15


For many operations that have an `axis` parameter, we can also pass `keepdims=True`. In that case the reduced axes do not disappear in the result; instead they are kept as axes with lengths of 1. This can be convenient when we want to use the result for shape alignment (broadcasting). Broadcasting is discussed in the second half of this chapter.


In [ ]:
sum0 = np.sum(arr2d, axis=0)
print("np.sum(arr2d, axis=0).shape: ", sum0.shape)
print("np.sum(arr2d, axis=0): ", sum0)

sum0_keepdims = np.sum(arr2d, axis=0, keepdims=True)
print("\nnp.sum(arr2d, axis=0, keepdims=True).shape: ", sum0_keepdims.shape)
print("np.sum(arr2d, axis=0, keepdims=True): ", sum0_keepdims)

sum01 = np.sum(arr2d, axis=(0,1))
print("\nnp.sum(arr2d, axis=(0,1)): ", sum01)    # scalar

sum01_keepdims = np.sum(arr2d, axis=(0,1), keepdims=True)
print("\nnp.sum(arr2d, axis=(0,1), keepdims=True).shape: ", sum01_keepdims.shape)
print("np.sum(arr2d, axis=(0,1), keepdims=True): ", sum01_keepdims)


np.sum(arr2d, axis=0).shape:  (3,)
np.sum(arr2d, axis=0):  [3 5 7]

np.sum(arr2d, axis=0, keepdims=True).shape:  (1, 3)
np.sum(arr2d, axis=0, keepdims=True):  [[3 5 7]]

np.sum(arr2d, axis=(0,1)):  15

np.sum(arr2d, axis=(0,1), keepdims=True).shape:  (1, 1)
np.sum(arr2d, axis=(0,1), keepdims=True):  [[15]]


Some additional examples of the functions introduced earlier, now using the `axis` parameter:


In [ ]:
arr2d = np.arange(6, dtype=np.int32).reshape(2,3)

print("The 2D 'arr2d' array:\n", arr2d)
print("   ... its shape is", arr2d.shape)

print("\nnp.prod(arr2d, axis=0): ", np.prod(arr2d, axis=0))
print("\nnp.mean(arr2d, axis=1): ", np.mean(arr2d, axis=1))
print("\nnp.std(arr2d, axis=0): ", np.std(arr2d, axis=0))
print("\nnp.var(arr2d, axis=1): ", np.var(arr2d, axis=1))
print("\nnp.amin(arr2d, axis=0): ", np.amin(arr2d, axis=0))
print("\nnp.amax(arr2d, axis=1): ", np.amax(arr2d, axis=1))


The 2D 'arr2d' array:
 [[0 1 2]
 [3 4 5]]
   ... its shape is (2, 3)

np.prod(arr2d, axis=0):  [ 0  4 10]

np.mean(arr2d, axis=1):  [1. 4.]

np.std(arr2d, axis=0):  [1.5 1.5 1.5]

np.var(arr2d, axis=1):  [0.66666667 0.66666667]

np.amin(arr2d, axis=0):  [0 1 2]

np.amax(arr2d, axis=1):  [2 5]


The `np.argmax()` and `np.argmin()` operations can be used similarly with the `axis` parameter: they search for the indices of the maximum/minimum values within each slice of the sliced array. The returned indices index the axis given in the `axis` parameter.


In [ ]:
print("\nnp.argmax(arr2d, axis=0): ", np.argmax(arr2d, axis=0))
print("\nnp.argmax(arr2d, axis=1): ", np.argmax(arr2d, axis=1))

print("\nnp.argmin(arr2d, axis=0): ", np.argmin(arr2d, axis=0))
print("\nnp.argmin(arr2d, axis=1): ", np.argmin(arr2d, axis=1))

# print("\nnp.argmin(arr2d, axis=0): ", np.argmax(arr2d, axis=(0,1)))   # cannot specify multiple axes


np.argmax(arr2d, axis=0):  [1 1 1]

np.argmax(arr2d, axis=1):  [2 2]

np.argmin(arr2d, axis=0):  [0 0 0]

np.argmin(arr2d, axis=1):  [0 0]


Specifying multiple axes is not supported for `np.argmax()` and `np.argmin()`, because flattened indices that index several (but not all) axes would be difficult to interpret. If we want to find the index of the maximum/minimum element over multiple axes, we need to merge the relevant axes into a single axis via a reshape.

Chaining multiple `np.argmax()` (or `np.argmin()`) operations does not solve the task described above, because the second operation in the chain no longer searches for a maximum/minimum among the array elements, but among the indices returned by the first operation:


In [ ]:
arr2d = np.arange(6, dtype=np.int32).reshape(2,3)

argmax1 = arr2d.argmax(axis=1)
print("arr2d.argmax(axis=1): ", argmax1)

argmax01_bad = arr2d.argmax(axis=1).argmax(axis=0)
print("\narr2d.argmax(axis=1).argmax(axis=0): ", argmax01_bad)   # NOT argmax along multiple axes

arr2d.argmax(axis=1):  [2 2]

arr2d.argmax(axis=1).argmax(axis=0):  0


_**Note:** Although in machine learning data can often be arranged into multi-dimensional arrays, there are some cases where the data length varies along one or more axes. Weather time series provide a good example for this issue. When grouping days into months: different months contain a different number of days. A naive solution would loop over months and process each month-specific array in the loop body. This is clearly not ideal, especially when the loop performs a large number of iterations._

_A potentially more efficient solution is to store all data in a single array whose relevant axis length is set to the longest month, and fill the missing days with a constant for shorter months. For floating point arrays this filler value can be `np.nan` (Not-a-Number). This choice is often advantageous because NumPy defines "NaN" variants of many functions (for example, `np.mean()` -> `np.nanmean()`), which ignore NaN elements during processing. This allows axis-wise operations on variable-length sequences and on masked arrays._

_Another approach is to use `np.ufunc.at()`, which provides group-by-like semantics. That is: the grouping of elements based on some conditions or labels, then, applying the same operation at once on each group of elements independently. NumPy does not offer a particularly large selection of group-by style operations, so it may be worth looking at, for example, the Pandas package, if you need similar functionality._


## Matching the shape of arrays: Broadcasting


In the previous chapter we already discussed element-wise operations on arrays. Pairwise addition of elements with the same indices in two arrays was a good example. To perform element-wise operations, we obviously need arrays of the same size/shape.

However, we may also encounter cases where we want to add a vector of the appropriate size to every row of a matrix. In other words, for a 2D array `a` we would like to execute `a[i, :] += v` for every index `i` along axis #0.
<br>$\ $</br>

<img src='../_files/nb_im/numpy_broadcast1-br2_op_only.png' width='50%'>

The straightforward, non-vectorized solution would use a loop:


In [ ]:
# INEFFICEINT SOLUTION: Python loop

a = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 2D 'a' array:\n", a)
print("   ... its shape is", a.shape)

v = np.array([10,20,30], dtype=np.int32)
print("\nThe 1D 'v' array:\n", v)
print("   ... its shape is", v.shape)

for i in range(a.shape[0]):
  a[i,:] += v

print("\nThe 'a' array with its rows increased by vector 'v':\n", a)

The 2D 'a' array:
 [[0 1 2]
 [3 4 5]]
   ... its shape is (2, 3)

The 1D 'v' array:
 [10 20 30]
   ... its shape is (3,)

The 'a' array with its rows increased by vector 'v':
 [[10 21 32]
 [13 24 35]]


Our goal this time, however, is to try to rely on NumPy features and solve the same problem with code that is both shorter and more efficient.

To eliminate the loop and still perform element-wise operations with arrays, we would need to somehow bring the arrays to the same shape, because only then can we match up elements with the same indices and add them pairwise.
<br>$\ $</br>

<img src='../_files/nb_im/numpy_broadcast1-br2_tile.png' width='50%'>


The NumPy `np.tile()` operation can be used to create a larger array by repeating the input array along the specified axes. Put differently, it "tiles" the input array to cover a larger array:


In [ ]:
# INEFFICEINT SOLUTION: np.tile() instead of proper broadcasting

a = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 2D 'a' array:\n", a)
print("   ... its shape is", a.shape)

v = np.array([10,20,30], dtype=np.int32)
print("\nThe 1D 'v' array:\n", v)
print("   ... its shape is", v.shape)

v_2d = v.reshape(1, v.shape[0])
print("\nThe 'v' array reshaped to 2D:\n", v_2d)
print("   ... its shape is", v_2d.shape)

v_tiled = np.tile(v_2d, (a.shape[0], 1))
print("\nThe tiled 2D 'v' array:\n", v_tiled)
print("   ... its shape is", v_tiled.shape)

a += v

print("\nThe 'a' array with its rows increased by the tiled vector 'v':\n", a)

The 2D 'a' array:
 [[0 1 2]
 [3 4 5]]
   ... its shape is (2, 3)

The 1D 'v' array:
 [10 20 30]
   ... its shape is (3,)

The 'v' array reshaped to 2D:
 [[10 20 30]]
   ... its shape is (1, 3)

The tiled 2D 'v' array:
 [[10 20 30]
 [10 20 30]]
   ... its shape is (2, 3)

The 'a' array with its rows increased by the tiled vector 'v':
 [[10 21 32]
 [13 24 35]]


As we can see above, `np.tile()` can only repeat the content of an array along axes that already exist. Therefore, as a first step we need to reshape the vector from `(3,)` to `(1, 3)`, because we want to match it to the shape of the matrix `a` and repeat its contents along `a`'s axis #0. After converting the vector to 2D, we can use `np.tile()` to repeat it twice along axis #0, obtaining shape `(2, 3)`. Since the two operands of the element-wise addition now have the same shape, the operation can be performed.

**Physically repeating the vector is an obviously unnecessary use of CPU time and memory.** Consider the case where we want to add the same constant to every element of an array with millions of elements. Following this logic, we would store the constant in a one-element array, allocate a new multi-million element array, and overwrite every element with the same constant, purely to make the second argument of the addition the same size as the original array. It is clear that, since the arguments of the addition are only read, it would be sufficient to use a special iterator over the one-element array that makes it **appear** to have millions of elements, while in reality it repeatedly reads and returns that single element as many times as necessary. In the matrix and row-vector example, such an iterator would know that the vector has a real extent along one axis (#1), but must be virtually repeated along the other axis (#0).

Fortunately, NumPy provides this functionality automatically, under the name **broadcasting**. Because of broadcasting, we do not need to construct exactly matching shapes for the two arguments of binary element-wise operations. Instead, it is enough if the two argument shapes are compatible ("broadcastable") with each other, meaning they satisfy certain compatibility rules that NumPy uses to infer the common shape needed to perform the operation.

If we inspect and run the following code block, it becomes clear that using `np.tile()` was unnecessary:


In [ ]:
# EFFICEINT SOLUTION: relying on broadcasting

a = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 2D 'a' array:\n", a)
print("   ... its shape is", a.shape)

v = np.array([10,20,30], dtype=np.int32)
print("\nThe 1D 'v' array:\n", v)
print("   ... its shape is", v.shape)

a += v

print("\nThe 'a' array with its rows increased by the vector 'v':\n", a)

The 2D 'a' array:
 [[0 1 2]
 [3 4 5]]
   ... its shape is (2, 3)

The 1D 'v' array:
 [10 20 30]
   ... its shape is (3,)

The 'a' array with its rows increased by the vector 'v':
 [[10 21 32]
 [13 24 35]]


The shape `(2, 3)` of array `a` and the shape `(3,)` of vector `v` are compatible (broadcastable). In this case, the operation `a + v` does exactly what we wanted.

What happens exactly?

- First, if the two shapes do not have the same length (that is, the number of dimensions differs), **the shape of the lower-dimensional array is left-padded with ones**.
- Next, NumPy checks whether the two shapes are compatible. **When matching shapes, data along axes of length 1 can be virtually repeated as many times as needed so that the axis length matches the length of the corresponding axis of the other array.** The axis lengths at matching positions are checked pairwise:
   - If the two axis lengths are not equal and neither is 1, the shapes are incompatible and NumPy raises an error.
   - If the two axis lengths are not equal but one of them is 1, the array is virtually repeated along the length-1 axis.
   - If the two axis lengths are equal, nothing needs to be done.

So in our example, the shape `(3,)` of vector `v` is first expanded by a leading 1 to `(1, 3)`, so its number of dimensions matches that of matrix `a`. Following the rules, shapes `(2, 3)` and `(1, 3)` are compatible, by virtually repeating `v` along axis #0 twice.

The virtual repetitions of the broadcasting mechanism do not copy elements in memory, so they are much more efficient than tiling with `np.tile()`.
<br>$\ $</br>

<img src='../_files/nb_im/numpy_broadcast1-br2.png' width='50%'>

Virtual repetitions are implemented by the special iterators that underpin N-dimensional arrays. In short, for the statement `a += v`, the addition operation does not receive the original `v` array as an argument, but rather a **view** of `v` whose shape has been broadcast to match `a`.

Now let's try to add a column vector to the matrix!


In [ ]:
a = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 2D 'a' array:\n", a)
print("   ... its shape is", a.shape)

v = np.array([10,20], dtype=np.int32)
print("\nThe 1D 'v' array:\n", v)
print("   ... its shape is", v.shape)

# a += v    # ValueError: operands could not be broadcast together with shapes (2,3) (2,) (2,3)


The 2D 'a' array:
 [[0 1 2]
 [3 4 5]]
   ... its shape is (2, 3)

The 1D 'v' array:
 [10 20]
   ... its shape is (2,)


In this case, if we were to run `a += v`, NumPy would raise the following error:

`ValueError: operands could not be broadcast together with shapes (2,3) (2,) (2,3)`

What is the problem?

If we look at the broadcasting rules again, we can see that the shape `(2, 3)` of array `a` and the shape `(2,)` of vector `v` are not compatible. After automatic padding to the same number of dimensions, the two shapes would be `(2, 3)` and `(1, 2)`. Along axis #1 the axis lengths (3 and 2) differ and both are greater than 1, so it is not clear how a matching axis length could be obtained.
<br>$\ $</br>

<img src='../_files/nb_im/numpy_broadcast1-br3_err.png' width='50%'>

In this particular case we might expect NumPy to infer that the length-2 vector should be aligned with the length-2 axis #0 of the matrix, but for example with arrays of shapes `(2, 2, 3)` and `(2,)` it would no longer be unambiguous whether we want to repeat the vector along axis #0 or axis #1. Therefore, **NumPy expects us to reshape the vector so that the axes we want to match end up in the same positions**.

That is, since we want the length of `v` to match the length 2 of axis #0 of array `a`, we need to reshape the vector to `(2, 1)`. This way its shape will no longer be automatically left-padded with ones:
<br>$\ $</br>

<img src='../_files/nb_im/numpy_broadcast1-br3.png' width='50%'>

In code, changing the shape of vector `v` can be done for example with `np.reshape()` or `ndarray.reshape()`:


In [ ]:
# EFFICIENT SOLUTION: relying on broadcasting

a = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 2D 'a' array:\n", a)
print("   ... its shape is", a.shape)

v = np.array([10,20], dtype=np.int32)
print("\nThe 1D 'v' array:\n", v)
print("   ... its shape is", v.shape)

v_2d = v.reshape(v.shape[0], 1)
print("\nThe 'v' array reshaped to 2D:\n", v_2d)
print("   ... its shape is", v_2d.shape)

a += v_2d

print("\nThe 'a' array with its columns increased by the vector 'v':\n", a)

The 2D 'a' array:
 [[0 1 2]
 [3 4 5]]
   ... its shape is (2, 3)

The 1D 'v' array:
 [10 20]
   ... its shape is (2,)

The 'v' array reshaped to 2D:
 [[10]
 [20]]
   ... its shape is (2, 1)

The 'a' array with its columns increased by the vector 'v':
 [[10 11 12]
 [23 24 25]]


An even more concise syntax for creating new axes of length 1 is to use indexing with `None` (or `np.newaxis`):


In [ ]:
a = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 2D 'a' array:\n", a)
print("   ... its shape is", a.shape)

v = np.array([10,20], dtype=np.int32)
print("\nThe 1D 'v' array:\n", v)
print("   ... its shape is", v.shape)

a += v[:,None]      # v[:,None] or v[:,np.newaxis] is equivalent to v.reshape(v.shape[0], 1) in this case

print("\nThe 'a' array with its columns increased by the vector 'v':\n", a)

The 2D 'a' array:
 [[0 1 2]
 [3 4 5]]
   ... its shape is (2, 3)

The 1D 'v' array:
 [10 20]
   ... its shape is (2,)

The 'a' array with its columns increased by the vector 'v':
 [[10 11 12]
 [23 24 25]]


Finally, an expression like `a + 1`, discussed earlier, also uses the broadcasting mechanism to perform the element-wise `+` operation. In this case there is an additional preliminary step before broadcasting, because the literal `1` is a Python `int` object. Since broadcasting requires two (or more) arrays as input, NumPy wraps the constant `1` into a newly created temporary array of shape `(1,)`.

After that, automatic broadcasting can proceed according to the rules: the constant array of shape `(1,)` is padded to match the number of dimensions of the `(2, 3)` array (new shape `(1, 1)`), and then it is virtually repeated along both axes to take on the common shape `(2, 3)`.
<br>$\ $</br>

<img src='../_files/nb_im/numpy_broadcast1-br1.png' width='50%'>

Just like in the row-vector case, there is no need to manipulate the shape by hand here either; broadcasting is performed automatically:


In [ ]:
a = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 2D 'a' array:\n", a)
print("   ... its shape is", a.shape)

a += 1

print("\nThe 'a' array with each element increased by 1:\n", a)

The 2D 'a' array:
 [[0 1 2]
 [3 4 5]]
   ... its shape is (2, 3)

The 'a' array with each element increased by 1:
 [[1 2 3]
 [4 5 6]]


In the examples above, broadcasting always involved repeating only one of the arrays along the relevant axis. If we read the broadcasting rules above carefully, we can see that this is not actually a requirement. **There are cases where we bring two (or more) arrays to a new common shape, and doing so requires virtual repetition along some axis for each array.**

This most often comes up in tasks where **we want to construct all possible pairings from the elements of two arrays** and then process those pairs independently using element-wise operations. The figure below shows exactly such a case: we pair the elements of two 1D arrays according to all possible combinations and multiply the members of each pair. For broadcasting to work this way, the extents of the two 1D arrays must be matched along different axes, so we add a new axis to one of the arrays, placed in the last position.

In this specific case we can also say that we turned one of the two row vectors into a column vector and then multiplied it with the row vector, producing an outer product. Note that the operation in the example below is element-wise multiplication. The shape of the result is determined by NumPy's broadcasting rules. This should not be confused with matrix multiplication, even if in special cases the results maybe the same.
<br>$\ $</br>


<img src='../_files/nb_im/numpy_broadcast1-br4.png' width='50%'>

In the block below we create a 5x5 multiplication table using broadcasting and the `*` operator:


In [ ]:
a = np.arange(5, dtype=np.int32) + 1
print("The 'a' array:\n", a)
print("   ... its shape is", a.shape)

mult_table = a * a[:,None]            # (5,) vs (5, 1) broadcasting -> (1, 5) vs (5, 1) ->  (5, 5) vs (5, 5)
print("\nThe 'mult_table' array:\n", mult_table)
print("   ... its shape is", mult_table.shape)

The 'a' array:
 [1 2 3 4 5]
   ... its shape is (5,)

The 'mult_table' array:
 [[ 1  2  3  4  5]
 [ 2  4  6  8 10]
 [ 3  6  9 12 15]
 [ 4  8 12 16 20]
 [ 5 10 15 20 25]]
   ... its shape is (5, 5)


## Swapping axis order


There are several reasons why we might want to change the axis order of arrays. For example, data coming from different sources may use different axis conventions. Images, for instance, are sometimes stored in `(size_x, size_y)` order, but the more common convention is matrix-style row-column order, `(size_y, size_x)`. If we load images in one format but the operations we apply expect the other format, we need to change the axis order.

If we only want to swap two axes, we can use the `np.swapaxes()` function:


In [ ]:
a = np.arange(12).reshape((2,2,3))

print("The 'a' array:\n", a)
print("   ... its shape is", a.shape)

a1 = np.swapaxes(a, 0, 1)

print("\nThe 'np.swapaxes(a, 0, 1)' array:\n", a1)
print("   ... its shape is", a1.shape)

a2 = np.swapaxes(a, 0, 2)

print("\nThe 'np.swapaxes(a, 0, 2)' array:\n", a2)
print("   ... its shape is", a2.shape)

a3 = np.swapaxes(a, 1, 2)

print("\nThe 'np.swapaxes(a, 1, 2)' array:\n", a3)
print("   ... its shape is", a3.shape)

The 'a' array:
 [[[ 0  1  2]
  [ 3  4  5]]

 [[ 6  7  8]
  [ 9 10 11]]]
   ... its shape is (2, 2, 3)

The 'np.swapaxes(a, 0, 1)' array:
 [[[ 0  1  2]
  [ 6  7  8]]

 [[ 3  4  5]
  [ 9 10 11]]]
   ... its shape is (2, 2, 3)

The 'np.swapaxes(a, 0, 2)' array:
 [[[ 0  6]
  [ 3  9]]

 [[ 1  7]
  [ 4 10]]

 [[ 2  8]
  [ 5 11]]]
   ... its shape is (3, 2, 2)

The 'np.swapaxes(a, 1, 2)' array:
 [[[ 0  3]
  [ 1  4]
  [ 2  5]]

 [[ 6  9]
  [ 7 10]
  [ 8 11]]]
   ... its shape is (2, 3, 2)


**The array produced by swapping axes is actually a view of the original array**, so writing to the swapped-axis array also modifies the original array. If we do not want that, we must copy the resulting view with `np.copy()` or `ndarray.copy()`. When swapping axes, only the behavior of the iterators that give the array its multi-dimensional interpretation changes; the underlying memory layout of the elements does not change.

We can apply `np.swapaxes()` multiple times in sequence to achieve the desired behavior, but it may be simpler to call `np.transpose()`, which lets us reorder the axes of an array arbitrarily:


In [ ]:
a = np.arange(12).reshape((2,2,3))

print("The 'a' array:\n", a)
print("   ... its shape is", a.shape)

a1 = np.transpose(a, (1, 2, 0))

print("\nThe 'a' array:\n", a1)
print("   ... its shape is", a1.shape)

The 'a' array:
 [[[ 0  1  2]
  [ 3  4  5]]

 [[ 6  7  8]
  [ 9 10 11]]]
   ... its shape is (2, 2, 3)

The 'a' array:
 [[[ 0  6]
  [ 1  7]
  [ 2  8]]

 [[ 3  9]
  [ 4 10]
  [ 5 11]]]
   ... its shape is (2, 3, 2)


If we do not provide an axis order to `np.transpose()`, it simply reverses the current axis order. In 2D this is identical to the matrix transpose operation. In this form, `np.transpose()` returns a view that is equivalent to the `ndarray.T` attribute.


## Linear algebra operations


With NumPy we can also perform linear algebra operations. Among these, the most important are the dot product and matrix multiplication.

In the simplest 1D case, the dot product is implemented by the `np.dot()` function. The dot product of two vectors is therefore:


In [ ]:
a = np.arange(6, dtype=np.float32)
b = np.arange(6, dtype=np.float32) + 1

print("The 'a' array:\n", a)
print("The 'b' array:\n", b)
print("   ... their shape is:", a.shape, b.shape)

dot_prod = np.dot(a, b)

print("\nTheir dot product is:", dot_prod)   # a scalar

The 'a' array:
 [0. 1. 2. 3. 4. 5.]
The 'b' array:
 [1. 2. 3. 4. 5. 6.]
   ... their shape is: (6,) (6,)

Their dot product is: 70.0


For 2D arrays, both `np.dot()` and `np.matmul()` perform the standard matrix multiplication. The matrix multiplication operator `@` also calls `np.matmul()`:


In [ ]:
a = np.arange(6, dtype=np.float32).reshape((2, 3))
b = np.arange(6, dtype=np.float32).reshape((3, 2))

print("The 'a' array:\n", a)
print("The 'b' array:\n", b)
print("   ... their shape is:", a.shape, b.shape)

mat_prod1 = np.dot(a, b)
mat_prod2 = np.matmul(a, b)
mat_prod3 = a @ b

print("\nnp.dot(a, b) is:\n", mat_prod1)    # a 2x2 matrix (with shape (2, 2))
print("np.matmul(a, b) is:\n", mat_prod2)   # a 2x2 matrix (with shape (2, 2))
print("a @ b is:\n", mat_prod3)             # operator alias for np.matmul()

The 'a' array:
 [[0. 1. 2.]
 [3. 4. 5.]]
The 'b' array:
 [[0. 1.]
 [2. 3.]
 [4. 5.]]
   ... their shape is: (2, 3) (3, 2)

np.dot(a, b) is:
 [[10. 13.]
 [28. 40.]]
np.matmul(a, b) is:
 [[10. 13.]
 [28. 40.]]
a @ b is:
 [[10. 13.]
 [28. 40.]]


We may want to compute several dot products independently and at once, over pairs of vectors. In other words, we are looking for a "batched dot product" operation. In NumPy, `np.einsum()` makes it possible to define more complex operations built from multiplications and summations, using a notation based on Einstein summation. Multiple (n) independent dot products can be computed as `np.einsum('ij,ij->i', a, b)`, where both `a` and `b` have shape `(n, k)` and contain n vectors of length k. The downside of `np.einsum()` is that it is much less efficient than a simple `np.dot()` or `np.matmul()` call. `np.einsum()` relies on single-threaded, custom NumPy code, whereas `np.dot()` and `np.matmul()` typically call BLAS (Basic Linear Algebra Subprograms), which provides highly optimized and parallelized routines. If you can use PyTorch, on CPU or GPU, `torch.bmm()` is generally a more efficient choice than NumPy's solutions.


For transposing matrices we can use the already discussed `np.transpose()` function. However, its shorter form, the `ndarray.T` attribute, can be more expressive in code:


In [ ]:
a = np.arange(6, dtype=np.float32).reshape((2, 3))

print("The 'a' array:\n", a)
print("The 'a.T' array:\n", a.T)
print("   ... their shape is:", a.shape, a.T.shape)

mat_prod = np.matmul(a, a.T)

print("\nnp.matmul(a, a.T) is:\n", mat_prod)    # (2, 3) x (3, 2) -> (2, 2)

The 'a' array:
 [[0. 1. 2.]
 [3. 4. 5.]]
The 'a.T' array:
 [[0. 3.]
 [1. 4.]
 [2. 5.]]
   ... their shape is: (2, 3) (3, 2)

np.matmul(a, a.T) is:
 [[ 5. 14.]
 [14. 50.]]


Other important linear algebra operations:
- `np.linalg.norm()`: matrix or vector norm
- `np.eye()`: create an identity matrix of a given size
- `np.linalg.svd()`: singular value decomposition of a matrix (for square real matrices this matches the eigendecomposition)
- `np.linalg.det()`: determinant of a matrix
- `np.linalg.matrix_rank()`: rank of a matrix
- `np.linalg.inv()`: inverse of a matrix
- `np.linalg.pinv()`: pseudoinverse of a matrix (also called the Moore-Penrose generalized inverse; it gives the exact solution of the least-squares problem)
